In [ ]:
comm_dict = {'L1':3, 'L2':8, 'L3':3, 'L4':3}
bp_dict = {'L1':5, 'L2':5, 'L3':3, 'L4':8}
pipe_seq_localsgd_waittime(bp_dict, comm_dict,2)

In [1]:
# dreamddp step1: scheduling
def determine_comm_schedule(comm_dict, bp_dict, H):
    layers = list(comm_dict.keys())
    num_layers = len(layers)
    # Sort layers by their backward processing sequence (assuming L1, L2, L3, L4...)
    # layers.sort(key=lambda x: int(x[1:]), reverse=True)
    layers.reverse()
    # print(layers)
    dp_cache = {}

    def calculate_wait_time(index):
      wait_time = 0
      bp_index = index + 1
      rest = 0
      tmp_idx = 0
      for i in range(index,num_layers):
        left_bp_time = get_remaining_bptime(rest, bp_index)
        #print(f'left_bp_time:{left_bp_time}')
        if(i == bp_index):
          #print('Condition satisfied')
          bp_index += 1
          rest = 0
          left_bp_time = get_remaining_bptime(rest, bp_index)
          #print(f'left_bp_time:{left_bp_time}')
        #print(f'i:{i}. bp_index:{bp_index}')
        if comm_dict[layers[i]] >= left_bp_time:
          tmp_idx = i
          wait_time = comm_dict[layers[i]] - left_bp_time
          break
        else:
          fall_idx, rest = find_fallin(rest, i, bp_index)
          #print(f'fall_idx:{fall_idx}. rest:{rest}')
          bp_index = fall_idx
          continue

      for j in range(tmp_idx + 1, num_layers):
        wait_time += comm_dict[layers[j]]
      return wait_time

    def get_remaining_bptime(rest, index):
      if index == num_layers:
        return 0
      bp_sum = 0
      # print(f'index:{index} num_layers:{num_layers} rest:{rest}')
      for i in range(index, num_layers):
          # print(f'Available layers: {layers[i]} index:{i} bp_time:{bp_dict[layers[i]]}')
          bp_sum += bp_dict[layers[i]]

      if rest == 0:
        return bp_sum
      else:
        return bp_sum - (bp_dict[layers[index]]-rest)

    def find_fallin(rest, index, bp_index):
      #print(f'rest:{rest} index:{index} bp_index:{bp_index}')
      #print(f'Comm time:{comm_dict[layers[index]]} rest {rest}')
      if comm_dict[layers[index]] <= rest:
        return bp_index, rest-comm_dict[layers[index]]
      else:
        fall_idx = 0
        rest_time = 0
        if rest == 0:
          total = comm_dict[layers[index]]
          #print(f'Total left: {total}')
          sum = 0
          left_in_bp = 0
          for i in range(bp_index, num_layers):
            sum += bp_dict[layers[i]]
            if (total <= sum):
              fall_idx = i
              rest_time = sum-total
              if(rest_time == 0):
                fall_idx += 1
              break
        else:
          total = comm_dict[layers[index]] - rest
          #print(f'Total left: {total}')
          sum = 0
          left_in_bp = 0
          for i in range(bp_index+1, num_layers):
            sum += bp_dict[layers[i]]
            if (total <= sum):
              fall_idx = i
              rest_time = sum-total
              if(rest_time == 0):
                fall_idx += 1
              break
        return fall_idx, rest_time

# This function will be called in each iteration
    def find_optimal_comm(index, assigned_iterations, left_iterations):
        total_iterations = 1
        current_iteration = assigned_iterations
        total_wait_time = 0
        total_comm_num = 0
        assign_dict = {}
        bp_idx = index + 1
        #comm_idx = index
        rest = 0
        findex = 0
        # base case
        if(index == num_layers):
          return 0,{},0

        if left_iterations == 1:
          for i in range(index, num_layers):
            assign_dict[layers[i]] = current_iteration
          total_wait_time += calculate_wait_time(index)
          return total_wait_time, assign_dict, 1

        for i in range(index, num_layers):
          #print()
          #print(f'current i: {i}. bp index {bp_idx}')
          if i == bp_idx:
            bp_idx += 1

          #print(f'rest:{rest} bp+idx{bp_idx}')
          remaining = get_remaining_bptime(rest,bp_idx)
          #print(f'Remaining time:{remaining}')
          if (comm_dict[layers[i]]) > remaining:
            # case 1: exceed
            if total_comm_num == 0:
              wait_time = comm_dict[layers[i]] - remaining
              # print(f'bp_idx {bp_idx} Comm time: {comm_dict[layers[i]]} remain time {remaining} watitime{wait_time}')
              assign_dict[layers[i]] = current_iteration
              total_wait_time += wait_time
              #print(f'Successfully communicate layer {i} Total wait time: {total_wait_time}')
              # process the next layer in next iteration
              wait_time1, dict1, iter1= find_optimal_comm(i+1, current_iteration+1,left_iterations-1 )
              total_wait_time += wait_time1
              total_iterations += iter1
              for name in dict1.keys():
                assign_dict[name] = dict1[name]
              break
            else:
              wait_time = comm_dict[layers[i]] - remaining
              #print(f'Enterin the first subloop')
              wait_time1, dict1, iter1= find_optimal_comm(i+1, current_iteration+1, left_iterations -1 )
              #print(f'first subloop total wait time: {wait_time + wait_time1}')
              #print(f'Enterin the second subloop')
              wait_time2, dict2, iter2 = find_optimal_comm(i,current_iteration+1, left_iterations -1)
              #print(f'first subloop total wait time: {wait_time2}')
              if (wait_time + wait_time1 <= wait_time2):
                assign_dict[layers[i]] = current_iteration
                total_wait_time += (wait_time + wait_time1)
                total_iterations += iter1
                for name in dict1.keys():
                  assign_dict[name] = dict1[name]
              else:
                total_wait_time += wait_time2
                total_iterations += iter2
                for name in dict2.keys():
                  assign_dict[name] = dict2[name]
              break
          else:
            bp_idx, rest = find_fallin(rest, i, bp_idx)
            #print(f'current layer {i} comm falls in {bp_idx}. rest time: {rest}')
            assign_dict[layers[i]] = current_iteration
            total_comm_num += 1
            #print(f'Successfully communicate layer {i} within the duration')

        return total_wait_time, assign_dict, total_iterations



    total_wait_time, optimal_schedule, total_iterations = find_optimal_comm(0, 0,H)
    return total_wait_time, optimal_schedule, total_iterations



In [2]:
# dreamddp step2: fillin more layers
def fillin_more_layers(total_iterations, comm_dict, bp_dict, schedule):
  new_schedule = {}
  layers = list(comm_dict.keys())
  num_layers = len(layers)
  layers.reverse()
  for name in layers:
    new_schedule[name] = []
    new_schedule[name].append(schedule[name])

  def get_remaining_bptime(rest, index, des):
      if index == des + 1:
        return 0
      bp_sum = 0
      # print(f'index:{index} num_layers:{num_layers} rest:{rest}')
      for i in range(index, des+1):
          # print(f'Available layers: {layers[i]} index:{i} bp_time:{bp_dict[layers[i]]}')
          bp_sum += bp_dict[layers[i]]

      if rest == 0:
        return bp_sum
      else:
        return bp_sum - (bp_dict[layers[index]]-rest)

  def find_first_comm_layer(iteration):
    for index, name in enumerate(schedule.keys()):
      if schedule[name] == iteration:
        return index,name
  def find_fallin(rest, index, bp_index):
      #print(f'rest:{rest} index:{index} bp_index:{bp_index}')
      #print(f'Comm time:{comm_dict[layers[index]]} rest {rest}')
      if comm_dict[layers[index]] <= rest:
        return bp_index, rest-comm_dict[layers[index]]
      else:
        fall_idx = 0
        rest_time = 0
        if rest == 0:
          total = comm_dict[layers[index]]
          #print(f'Total left: {total}')
          sum = 0
          left_in_bp = 0
          for i in range(bp_index, num_layers):
            sum += bp_dict[layers[i]]
            if (total <= sum):
              fall_idx = i
              rest_time = sum-total
              if(rest_time == 0):
                fall_idx += 1
              break
        else:
          total = comm_dict[layers[index]] - rest
          #print(f'Total left: {total}')
          sum = 0
          left_in_bp = 0
          for i in range(bp_index+1, num_layers):
            sum += bp_dict[layers[i]]
            if (total <= sum):
              fall_idx = i
              rest_time = sum-total
              if(rest_time == 0):
                fall_idx += 1
              break
        return fall_idx, rest_time

  for iteration in range(total_iterations):
    # print()
    # print(f'Iteration: {iteration}')
    des,name = find_first_comm_layer(iteration)
    # print(f'First comm layer:{des} iteration:{iteration}')
    rest = 0
    bp_index = 1
    if(des == 0):
      print(f'No available position in the first iteration')
      continue
    for i in range(des):
      if (bp_index == i):
        # print('Add bp index')
        bp_index += 1
        rest = 0
      # print(f'comm index:{i} bp_index:{bp_index}')
      remaining = get_remaining_bptime(rest,bp_index,des)
      # print(f'Remaining time: {remaining}')
      if comm_dict[layers[i]] <= remaining:
        fall_idx, rest_time = find_fallin(rest,i,bp_index)
        bp_index = fall_idx
        rest = rest_time
        new_schedule[layers[i]].append(iteration)
        # print(f'Successfully fillin {layers[i]} index: {i}')
      else:
        break
  return new_schedule

In [3]:
def pipe_seq_localsgd_waittime(bp_dict, comm_dict,H):
  wait_list = []
  layers = list(comm_dict.keys())
  num_layers = len(layers)
  layers.reverse()
  if (len(layers) % H) == 0:
    layers_per_iter = int(len(layers) / H)
  else:
    layers_per_iter = int(len(layers) / H) + 1
  def find_fallin(rest, index, bp_index):
      #print(f'rest:{rest} index:{index} bp_index:{bp_index}')
      #print(f'Comm time:{comm_dict[layers[index]]} rest {rest}')
      if comm_dict[layers[index]] <= rest:
        return bp_index, rest-comm_dict[layers[index]]
      else:
        fall_idx = 0
        rest_time = 0
        if rest == 0:
          total = comm_dict[layers[index]]
          #print(f'Total left: {total}')
          sum = 0
          left_in_bp = 0
          for i in range(bp_index, num_layers):
            sum += bp_dict[layers[i]]
            if (total <= sum):
              fall_idx = i
              rest_time = sum-total
              if(rest_time == 0):
                fall_idx += 1
              break
        else:
          total = comm_dict[layers[index]] - rest
          #print(f'Total left: {total}')
          sum = 0
          left_in_bp = 0
          for i in range(bp_index+1, num_layers):
            sum += bp_dict[layers[i]]
            if (total <= sum):
              fall_idx = i
              rest_time = sum-total
              if(rest_time == 0):
                fall_idx += 1
              break
        return fall_idx, rest_time
  def get_remaining_bptime(rest, index):
      if index == num_layers:
        return 0
      bp_sum = 0
      # print(f'index:{index} num_layers:{num_layers} rest:{rest}')
      for i in range(index, num_layers):
          # print(f'Available layers: {layers[i]} index:{i} bp_time:{bp_dict[layers[i]]}')
          bp_sum += bp_dict[layers[i]]

      if rest == 0:
        return bp_sum
      else:
        return bp_sum - (bp_dict[layers[index]]-rest)
  def get_waittime(start_index, end_index):
    bp_idx = start_index + 1
    total_wait_time = 0
    rest = 0
    for i in range(start_index, end_index):
      if (i == bp_idx):
        bp_idx += 1
        rest = 0
      remaining = get_remaining_bptime(rest,bp_idx)
      if(comm_dict[layers[i]] >= remaining):
        wait_time = comm_dict[layers[i]] - remaining
        total_wait_time += wait_time
        for j in range(i+1, end_index):
          total_wait_time += comm_dict[layers[j]]
        return total_wait_time
      else:
        fall_idx, rest_time = find_fallin(rest, i, bp_idx)
        bp_idx = fall_idx
        rest = rest_time
    return total_wait_time

  for i in range(H):
    start_index = layers_per_iter * i
    end_index = min(start_index + layers_per_iter, len(layers))
    print(f'start index:{start_index} end_index:{end_index}')
    wt = get_waittime(start_index, end_index)
    wait_list.append(wt)
  return wait_list

In [4]:
comm_dict ={
    "transformer.wte": 0.1728140354156494,
    "transformer.wpe": 0.004974079132080078,
    "transformer.drop": 1.2969970703125e-05,
    "transformer.h.0.ln_1": 0.002202749252319336,
    "transformer.h.0.attn.c_attn": 0.009178066253662109,
    "transformer.h.0.attn.c_proj": 0.004050159454345703,
    "transformer.h.0.attn.attn_dropout": 1.1205673217773438e-05,
    "transformer.h.0.attn.resid_dropout": 9.965896606445312e-06,
    "transformer.h.0.ln_2": 0.0005535602569580079,
    "transformer.h.0.mlp.c_fc": 0.011739540100097656,
    "transformer.h.0.mlp.c_proj": 0.011766004562377929,
    "transformer.h.0.mlp.act": 1.4734268188476562e-05,
    "transformer.h.0.mlp.dropout": 1.049041748046875e-05,
    "transformer.h.1.ln_1": 0.0006047725677490234,
    "transformer.h.1.attn.c_attn": 0.008490753173828126,
    "transformer.h.1.attn.c_proj": 0.003826141357421875,
    "transformer.h.1.attn.attn_dropout": 1.1110305786132813e-05,
    "transformer.h.1.attn.resid_dropout": 1.0061264038085938e-05,
    "transformer.h.1.ln_2": 0.0005242347717285157,
    "transformer.h.1.mlp.c_fc": 0.012103986740112305,
    "transformer.h.1.mlp.c_proj": 0.011976528167724609,
    "transformer.h.1.mlp.act": 1.163482666015625e-05,
    "transformer.h.1.mlp.dropout": 1.0061264038085938e-05,
    "transformer.h.2.ln_1": 0.000792694091796875,
    "transformer.h.2.attn.c_attn": 0.009080028533935547,
    "transformer.h.2.attn.c_proj": 0.004201650619506836,
    "transformer.h.2.attn.attn_dropout": 1.087188720703125e-05,
    "transformer.h.2.attn.resid_dropout": 9.918212890625e-06,
    "transformer.h.2.ln_2": 0.0004822731018066406,
    "transformer.h.2.mlp.c_fc": 0.011906623840332031,
    "transformer.h.2.mlp.c_proj": 0.011779212951660156,
    "transformer.h.2.mlp.act": 1.1730194091796876e-05,
    "transformer.h.2.mlp.dropout": 9.822845458984376e-06,
    "transformer.h.3.ln_1": 0.00048646926879882815,
    "transformer.h.3.attn.c_attn": 0.008724641799926759,
    "transformer.h.3.attn.c_proj": 0.0040285587310791016,
    "transformer.h.3.attn.attn_dropout": 1.087188720703125e-05,
    "transformer.h.3.attn.resid_dropout": 9.822845458984376e-06,
    "transformer.h.3.ln_2": 0.0004761219024658203,
    "transformer.h.3.mlp.c_fc": 0.012328004837036133,
    "transformer.h.3.mlp.c_proj": 0.0119232177734375,
    "transformer.h.3.mlp.act": 1.1301040649414062e-05,
    "transformer.h.3.mlp.dropout": 1.0061264038085938e-05,
    "transformer.h.4.ln_1": 0.0004809856414794922,
    "transformer.h.4.attn.c_attn": 0.008501434326171875,
    "transformer.h.4.attn.c_proj": 0.003973531723022461,
    "transformer.h.4.attn.attn_dropout": 1.0728836059570312e-05,
    "transformer.h.4.attn.resid_dropout": 1.0013580322265625e-05,
    "transformer.h.4.ln_2": 0.0004837512969970703,
    "transformer.h.4.mlp.c_fc": 0.012636947631835937,
    "transformer.h.4.mlp.c_proj": 0.012026262283325196,
    "transformer.h.4.mlp.act": 1.1348724365234375e-05,
    "transformer.h.4.mlp.dropout": 1.0013580322265625e-05,
    "transformer.h.5.ln_1": 0.0006218433380126953,
    "transformer.h.5.attn.c_attn": 0.00897836685180664,
    "transformer.h.5.attn.c_proj": 0.004136180877685547,
    "transformer.h.5.attn.attn_dropout": 1.1110305786132813e-05,
    "transformer.h.5.attn.resid_dropout": 9.918212890625e-06,
    "transformer.h.5.ln_2": 0.000457000732421875,
    "transformer.h.5.mlp.c_fc": 0.01171126365661621,
    "transformer.h.5.mlp.c_proj": 0.012033557891845703,
    "transformer.h.5.mlp.act": 1.163482666015625e-05,
    "transformer.h.5.mlp.dropout": 9.965896606445312e-06,
    "transformer.h.6.ln_1": 0.0004745006561279297,
    "transformer.h.6.attn.c_attn": 0.008822870254516602,
    "transformer.h.6.attn.c_proj": 0.004080867767333985,
    "transformer.h.6.attn.attn_dropout": 1.0967254638671875e-05,
    "transformer.h.6.attn.resid_dropout": 9.822845458984376e-06,
    "transformer.h.6.ln_2": 0.000460052490234375,
    "transformer.h.6.mlp.c_fc": 0.012621355056762696,
    "transformer.h.6.mlp.c_proj": 0.012290716171264648,
    "transformer.h.6.mlp.act": 1.163482666015625e-05,
    "transformer.h.6.mlp.dropout": 9.870529174804687e-06,
    "transformer.h.7.ln_1": 0.0005825519561767578,
    "transformer.h.7.attn.c_attn": 0.008771800994873047,
    "transformer.h.7.attn.c_proj": 0.004177761077880859,
    "transformer.h.7.attn.attn_dropout": 1.10626220703125e-05,
    "transformer.h.7.attn.resid_dropout": 1.0061264038085938e-05,
    "transformer.h.7.ln_2": 0.0005350589752197265,
    "transformer.h.7.mlp.c_fc": 0.01255784034729004,
    "transformer.h.7.mlp.c_proj": 0.012088918685913086,
    "transformer.h.7.mlp.act": 1.1444091796875e-05,
    "transformer.h.7.mlp.dropout": 9.918212890625e-06,
    "transformer.h.8.ln_1": 0.00047955513000488283,
    "transformer.h.8.attn.c_attn": 0.008948850631713866,
    "transformer.h.8.attn.c_proj": 0.004285383224487305,
    "transformer.h.8.attn.attn_dropout": 1.0776519775390626e-05,
    "transformer.h.8.attn.resid_dropout": 9.822845458984376e-06,
    "transformer.h.8.ln_2": 0.0004711151123046875,
    "transformer.h.8.mlp.c_fc": 0.012849950790405273,
    "transformer.h.8.mlp.c_proj": 0.012043952941894531,
    "transformer.h.8.mlp.act": 1.1301040649414062e-05,
    "transformer.h.8.mlp.dropout": 9.965896606445312e-06,
    "transformer.h.9.ln_1": 0.0004572868347167969,
    "transformer.h.9.attn.c_attn": 0.008754920959472657,
    "transformer.h.9.attn.c_proj": 0.0040452003479003905,
    "transformer.h.9.attn.attn_dropout": 1.0919570922851562e-05,
    "transformer.h.9.attn.resid_dropout": 9.822845458984376e-06,
    "transformer.h.9.ln_2": 0.00043025016784667967,
    "transformer.h.9.mlp.c_fc": 0.01287703514099121,
    "transformer.h.9.mlp.c_proj": 0.012273597717285156,
    "transformer.h.9.mlp.act": 1.1157989501953124e-05,
    "transformer.h.9.mlp.dropout": 9.918212890625e-06,
    "transformer.h.10.ln_1": 0.00045137405395507814,
    "transformer.h.10.attn.c_attn": 0.008859348297119141,
    "transformer.h.10.attn.c_proj": 0.003995323181152343,
    "transformer.h.10.attn.attn_dropout": 1.0824203491210937e-05,
    "transformer.h.10.attn.resid_dropout": 9.870529174804687e-06,
    "transformer.h.10.ln_2": 0.0004390716552734375,
    "transformer.h.10.mlp.c_fc": 0.012881803512573241,
    "transformer.h.10.mlp.c_proj": 0.01248340606689453,
    "transformer.h.10.mlp.act": 1.1205673217773438e-05,
    "transformer.h.10.mlp.dropout": 9.918212890625e-06,
    "transformer.h.11.ln_1": 0.00045833587646484377,
    "transformer.h.11.attn.c_attn": 0.009063482284545898,
    "transformer.h.11.attn.c_proj": 0.004229879379272461,
    "transformer.h.11.attn.attn_dropout": 1.0919570922851562e-05,
    "transformer.h.11.attn.resid_dropout": 9.965896606445312e-06,
    "transformer.h.11.ln_2": 0.0006897449493408203,
    "transformer.h.11.mlp.c_fc": 0.01290440559387207,
    "transformer.h.11.mlp.c_proj": 0.011844730377197266,
    "transformer.h.11.mlp.act": 1.1348724365234375e-05,
    "transformer.h.11.mlp.dropout": 9.870529174804687e-06,
    "transformer.ln_f": 0.00048084259033203124,
    "lm_head": 0.18098649978637696
}


bp_dict ={
    "model.embed_tokens": 0.02171309240933122,
    "model.layers.0.self_attn.q_proj": 0.0004350138806748664,
    "model.layers.0.self_attn.k_proj": 0.0004306922013732209,
    "model.layers.0.self_attn.v_proj": 0.00042797505170449445,
    "model.layers.0.self_attn.o_proj": 0.00042759275984490056,
    "model.layers.0.self_attn.rotary_emb": 1.145642379234577e-05,
    "model.layers.0.mlp.gate_proj": 0.007708478247982332,
    "model.layers.0.mlp.up_proj": 0.007667736075390345,
    "model.layers.0.mlp.down_proj": 0.007646045465578978,
    "model.layers.0.mlp.act_fn": 1.1245409647623697e-05,
    "model.layers.0.input_layernorm": 8.116371330173537e-05,
    "model.layers.0.post_attention_layernorm": 6.183947639903803e-05,
    "model.layers.1.self_attn.q_proj": 0.0004268322867908697,
    "model.layers.1.self_attn.k_proj": 0.0004255196143840921,
    "model.layers.1.self_attn.v_proj": 0.00042444399033469716,
    "model.layers.1.self_attn.o_proj": 0.00042539629442938444,
    "model.layers.1.self_attn.rotary_emb": 1.0442459720304643e-05,
    "model.layers.1.mlp.gate_proj": 0.007695925646814807,
    "model.layers.1.mlp.up_proj": 0.007666871465485671,
    "model.layers.1.mlp.down_proj": 0.007643681833113747,
    "model.layers.1.mlp.act_fn": 1.0647992978150817e-05,
    "model.layers.1.input_layernorm": 8.225989067691497e-05,
    "model.layers.1.post_attention_layernorm": 6.386329387796337e-05,
    "model.layers.2.self_attn.q_proj": 0.00042486875906757926,
    "model.layers.2.self_attn.k_proj": 0.0004326557290965113,
    "model.layers.2.self_attn.v_proj": 0.0004250756625471444,
    "model.layers.2.self_attn.o_proj": 0.00042438096013562435,
    "model.layers.2.self_attn.rotary_emb": 1.0261590453400009e-05,
    "model.layers.2.mlp.gate_proj": 0.007697098556606249,
    "model.layers.2.mlp.up_proj": 0.007666313785246049,
    "model.layers.2.mlp.down_proj": 0.0076495477522926765,
    "model.layers.2.mlp.act_fn": 1.0708282733785695e-05,
    "model.layers.2.input_layernorm": 9.255710689500831e-05,
    "model.layers.2.post_attention_layernorm": 6.0389781820363014e-05,
    "model.layers.3.self_attn.q_proj": 0.0004245974551672223,
    "model.layers.3.self_attn.k_proj": 0.0004248125799771013,
    "model.layers.3.self_attn.v_proj": 0.00042393426785523864,
    "model.layers.3.self_attn.o_proj": 0.00042501811323494745,
    "model.layers.3.self_attn.rotary_emb": 1.0171155819947692e-05,
    "model.layers.3.mlp.gate_proj": 0.007697434260927397,
    "model.layers.3.mlp.up_proj": 0.0076699229492538275,
    "model.layers.3.mlp.down_proj": 0.007650097211201985,
    "model.layers.3.mlp.act_fn": 1.068224852112518e-05,
    "model.layers.3.input_layernorm": 8.183512194403287e-05,
    "model.layers.3.post_attention_layernorm": 6.053091465741738e-05,
    "model.layers.4.self_attn.q_proj": 0.00042589505513509113,
    "model.layers.4.self_attn.k_proj": 0.00042450702053376996,
    "model.layers.4.self_attn.v_proj": 0.000431382793119584,
    "model.layers.4.self_attn.o_proj": 0.00042370955149332684,
    "model.layers.4.self_attn.rotary_emb": 1.0184858037137438e-05,
    "model.layers.4.mlp.gate_proj": 0.007701087272030184,
    "model.layers.4.mlp.up_proj": 0.0076721002315652785,
    "model.layers.4.mlp.down_proj": 0.007646492157859364,
    "model.layers.4.mlp.act_fn": 1.0649363199869791e-05,
    "model.layers.4.input_layernorm": 7.993462441981524e-05,
    "model.layers.4.post_attention_layernorm": 5.922098269407777e-05,
    "model.layers.5.self_attn.q_proj": 0.00042377258169239965,
    "model.layers.5.self_attn.k_proj": 0.0004244727649907956,
    "model.layers.5.self_attn.v_proj": 0.0004238191692308448,
    "model.layers.5.self_attn.o_proj": 0.0004269994538405846,
    "model.layers.5.self_attn.rotary_emb": 1.0184858037137438e-05,
    "model.layers.5.mlp.gate_proj": 0.007699364903329432,
    "model.layers.5.mlp.up_proj": 0.007670539549027367,
    "model.layers.5.mlp.down_proj": 0.007643358460788069,
    "model.layers.5.mlp.act_fn": 1.0580852113921067e-05,
    "model.layers.5.input_layernorm": 8.551005659432246e-05,
    "model.layers.5.post_attention_layernorm": 6.296168798687814e-05,
    "model.layers.6.self_attn.q_proj": 0.00042788735751448005,
    "model.layers.6.self_attn.k_proj": 0.0004289232451340248,
    "model.layers.6.self_attn.v_proj": 0.00042495919370103154,
    "model.layers.6.self_attn.o_proj": 0.0004316965738932292,
    "model.layers.6.self_attn.rotary_emb": 1.017389626338564e-05,
    "model.layers.6.mlp.gate_proj": 0.007696392892420977,
    "model.layers.6.mlp.up_proj": 0.007671719309927404,
    "model.layers.6.mlp.down_proj": 0.007649059953360722,
    "model.layers.6.mlp.act_fn": 1.0569890340169271e-05,
    "model.layers.6.input_layernorm": 7.59390578872856e-05,
    "model.layers.6.post_attention_layernorm": 5.890994236387055e-05,
    "model.layers.7.self_attn.q_proj": 0.00042719128488124104,
    "model.layers.7.self_attn.k_proj": 0.000424076770914012,
    "model.layers.7.self_attn.v_proj": 0.0004251866505063813,
    "model.layers.7.self_attn.o_proj": 0.00042416446510402635,
    "model.layers.7.self_attn.rotary_emb": 1.01492322724441e-05,
    "model.layers.7.mlp.gate_proj": 0.007697716526601507,
    "model.layers.7.mlp.up_proj": 0.0076708122231494424,
    "model.layers.7.mlp.down_proj": 0.0076419950901776895,
    "model.layers.7.mlp.act_fn": 1.0608256548300557e-05,
    "model.layers.7.input_layernorm": 8.121989239221332e-05,
    "model.layers.7.post_attention_layernorm": 5.853724205630949e-05,
    "model.norm": 5.932100887956291e-05,
    "lm_head": 0.022227668214118344
}
for key in comm_dict.keys():
  comm_dict[key] *= 10

In [5]:
bp_sum = 0.0
comm_sum = 0.0
for key in comm_dict:
  # bp_sum+= bp_dict[key]
  comm_sum += comm_dict[key]
print(f'comm:{comm_sum}')
# print(f'bp:{bp_sum}')

comm:8.227286338806152


In [6]:
# Function to round dictionary values to specified decimal places
def round_dict_values(input_dict, decimal_places):
    return {key: round(value, decimal_places) for key, value in input_dict.items()}

# Round the values in both dictionaries to 3 decimal places
rounded_comm_dict_32 = round_dict_values(comm_dict, 8)
rounded_bp_dict_32 = round_dict_values(bp_dict, 8)

# Output the rounded dictionaries
#  rounded_comm_dict_32, rounded_bp_dict_32

In [7]:
wait_list = pipe_seq_localsgd_waittime(rounded_bp_dict_32, rounded_comm_dict_32,5)
wait_list

start index:0 end_index:25


KeyError: 'transformer.ln_f'

In [ ]:
H = 5
total_wait_time, schedule, total_iterations = determine_comm_schedule(rounded_comm_dict_32, rounded_bp_dict_32, H)
print("Schedule:", schedule)
print("Total Iterations:", total_iterations)
print("Total waittime:", total_wait_time)

Schedule: {'lm_head': 0, 'model.norm': 1, 'model.layers.7.post_attention_layernorm': 1, 'model.layers.7.input_layernorm': 1, 'model.layers.7.mlp.act_fn': 1, 'model.layers.7.mlp.down_proj': 1, 'model.layers.7.mlp.up_proj': 1, 'model.layers.7.mlp.gate_proj': 2, 'model.layers.7.self_attn.rotary_emb': 2, 'model.layers.7.self_attn.o_proj': 2, 'model.layers.7.self_attn.v_proj': 2, 'model.layers.7.self_attn.k_proj': 2, 'model.layers.7.self_attn.q_proj': 2, 'model.layers.6.post_attention_layernorm': 2, 'model.layers.6.input_layernorm': 2, 'model.layers.6.mlp.act_fn': 2, 'model.layers.6.mlp.down_proj': 2, 'model.layers.6.mlp.up_proj': 3, 'model.layers.6.mlp.gate_proj': 3, 'model.layers.6.self_attn.rotary_emb': 4, 'model.layers.6.self_attn.o_proj': 4, 'model.layers.6.self_attn.v_proj': 4, 'model.layers.6.self_attn.k_proj': 4, 'model.layers.6.self_attn.q_proj': 4, 'model.layers.5.post_attention_layernorm': 4, 'model.layers.5.input_layernorm': 4, 'model.layers.5.mlp.act_fn': 4, 'model.layers.5.mlp

In [ ]:
fillin_more_layers(total_iterations, rounded_comm_dict_32, rounded_bp_dict_32, schedule)

No available position in the first iteration


{'linear': [0, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14],
 'layer4.2.shortcut': [0, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14],
 'layer4.2.bn3': [0, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14],
 'layer4.2.conv3': [0],
 'layer4.2.bn2': [0],
 'layer4.2.conv2': [1],
 'layer4.2.bn1': [2],
 'layer4.2.conv1': [2],
 'layer4.1.shortcut': [2],
 'layer4.1.bn3': [2],
 'layer4.1.conv3': [2],
 'layer4.1.bn2': [3],
 'layer4.1.conv2': [3],
 'layer4.1.bn1': [4],
 'layer4.1.conv1': [4],
 'layer4.0.shortcut.1': [4],
 'layer4.0.shortcut.0': [5],
 'layer4.0.bn3': [6],
 'layer4.0.conv3': [6],
 'layer4.0.bn2': [6],
 'layer4.0.conv2': [7],
 'layer4.0.bn1': [8],
 'layer4.0.conv1': [8],
 'layer3.5.shortcut': [8],
 'layer3.5.bn3': [8],
 'layer3.5.conv3': [8],
 'layer3.5.bn2': [8],
 'layer3.5.conv2': [8],
 'layer3.5.bn1': [9],
 'layer3.5.conv1': [9],
 'layer3.4.shortcut': [9],
 'layer3.4.bn3': [9],
 'layer3.4.conv3': [9],
 'layer3.4.bn2': [9],
 'layer3.4.conv2': [9],
 'layer3.4.bn1': [9],
 'layer3.4.conv1': [9],
 'layer3.3.shortcut': [